# 06 — Scenario & Stress Simulation

Apply base, adverse-credit, and high-prepayment macro scenarios. Compare projected delinquency, default, and prepayment rates. Show segment-level impacts.

In [ ]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

results = json.load(open('../reports/scenario/scenario_results.json'))
print('Scenarios:', list(results.keys()))


## Aggregate Projections

In [ ]:
rows = []
for name, res in results.items():
    agg = res.get('aggregate_projections', {})
    rows.append({'scenario': name, **{k: round(v,4) for k,v in agg.items() if isinstance(v,(int,float))}})
agg_df = pd.DataFrame(rows).set_index('scenario')
print(agg_df.to_string())


## Side-by-Side Rate Comparison

In [ ]:
rate_cols = [c for c in agg_df.columns if '_rate' in c or '_flag_rate' in c][:4]
fig, ax = plt.subplots(figsize=(10, 4))
agg_df[rate_cols].T.plot(kind='bar', ax=ax,
                          color=['#4C9BE8','#E87070','#59B86E'])
ax.set_title('Projected Rates by Scenario')
ax.set_ylabel('Rate')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Scenario')
plt.tight_layout()
plt.savefig('../reports/scenario_comparison.png', dpi=150)
plt.show()


## Segment-Level Impacts (base vs adverse)

In [ ]:
base_seg    = results.get('base', {}).get('segment_projections', {})
adverse_seg = results.get('adverse_credit', {}).get('segment_projections', {})
if base_seg and adverse_seg:
    seg_rows = []
    for seg in set(list(base_seg.keys())[:6]) & set(adverse_seg.keys()):
        b = base_seg[seg]
        a = adverse_seg[seg]
        seg_rows.append({'segment': seg,
                         'base_default_rate': round(b.get('next_12m_default_flag_rate', float('nan')), 4),
                         'adverse_default_rate': round(a.get('next_12m_default_flag_rate', float('nan')), 4)})
    seg_df = pd.DataFrame(seg_rows).sort_values('adverse_default_rate', ascending=False)
    print(seg_df.to_string(index=False))
else:
    print('Segment projections not available in scenario results.')
